# GörEndir — YouTube Video & Subtitle Downloader (LOCAL Jupyter)

This notebook is for running **GörEndir** on your **local computer** with Jupyter Notebook / JupyterLab.

## Features
- Download single videos or entire playlists
- Multi-language subtitle extraction (SRT + TXT)
- `Dict[str, int]` input format: `{"url": start_number}`
- `reverse_download=True`: reverse playlist order (last video → #1)
- `skip_download=True`: download only subtitles, skip video file
- `playlist_end=N`: limit number of videos
- **SSL workaround** for local machines behind corporate proxy / antivirus SSL inspection

## ⚠️ Local-specific notes
1. **SSL fix** — Step 2 monkey-patches `yt-dlp` and `requests` to skip SSL verification. This is needed when your network injects self-signed certificates (corporate proxy, antivirus HTTPS scanning, Fiddler/Charles, etc.).
2. **Deno** — only needed if you download video files (`skip_download=False`). For subtitle-only downloads, Deno is not required.
3. **Cookies** — YouTube may still require cookies even on local machines.

Common local errors this notebook fixes:
- `[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain`
- `No supported JavaScript runtime could be found`
- `n challenge solving failed`
- `Sign in to confirm you're not a bot`

## Step 1: Install GörEndir + Dependencies

In [ ]:
# Install / upgrade gorendir from GitHub + yt-dlp + certifi
!pip install -q --upgrade yt-dlp certifi
!pip install -q --upgrade --force-reinstall git+https://github.com/MammadTavakoli/gorendir.git
!yt-dlp --version

## Step 2: SSL Fix (RUN BEFORE importing gorendir)

This step monkey-patches `yt-dlp` and `requests` so that **all** HTTPS connections skip certificate verification. It works **regardless of the gorendir version** installed — no need for a `verify_ssl` parameter in `YouTubeDownloader`.

### When do you need this?
Run this cell if you see:
```
[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain
```

### Why does this error happen on local but not on Colab?
Colab connects to YouTube directly. Your local machine probably has one of:
1. **Corporate / enterprise proxy** doing SSL inspection (Zscaler, Fortinet, Cisco, Palo Alto, etc.)
2. **Antivirus** with HTTPS/Web Shield scanning (Kaspersky, ESET, Avast, Bitdefender, etc.)
3. **Fiddler / Charles / mitmproxy** running in the background
4. **Missing CA certificates** on Windows (common with Python 3.13)

### Security note
Disabling SSL verification is **insecure on public networks**. Only do this on a trusted local machine. If your company provides a root CA certificate, the safer alternative is:
```python
import os, certifi
os.environ['REQUESTS_CA_BUNDLE'] = '/path/to/company-root-ca.pem'
os.environ['SSL_CERT_FILE'] = '/path/to/company-root-ca.pem'
```

In [ ]:
# === SSL VERIFICATION DISABLE — run this BEFORE `from gorendir import ...` ===
# IMPORTANT: yt-dlp's YoutubeDL.__init__ takes `params` (a dict) as its first
# positional argument, NOT keyword arguments. So we must inject
# `nocheckcertificate` INTO the params dict, not as a kwarg.
import yt_dlp
import requests
import urllib3

# 1) Force yt-dlp to skip certificate checks on every YoutubeDL instance.
#    This covers all of gorendir's internal yt-dlp calls (extract_info + download).
_orig_ydl_init = yt_dlp.YoutubeDL.__init__
def _patched_ydl_init(self, params=None, **kwargs):
    if params is None:
        params = {}
    if isinstance(params, dict):
        params.setdefault('nocheckcertificate', True)
    return _orig_ydl_init(self, params, **kwargs)
yt_dlp.YoutubeDL.__init__ = _patched_ydl_init

# 2) Force every requests.Session to default to verify=False.
#    This covers gorendir's youtube_transcript_api session.
_orig_sess_init = requests.Session.__init__
def _patched_sess_init(self, *args, **kwargs):
    _orig_sess_init(self, *args, **kwargs)
    self.verify = False
requests.Session.__init__ = _patched_sess_init

# 3) Silence the urllib3 InsecureRequestWarning noise.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 4) (Optional, safer alternative) If you have a company root CA, use it instead:
#    import os, certifi
#    os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
#    os.environ['SSL_CERT_FILE'] = certifi.where()

print('✅ SSL verification disabled for yt-dlp and requests (local workaround active).')

## Step 3: Install Deno (only needed for video downloads)

yt-dlp needs Deno to solve YouTube's JavaScript `n`-challenge.
- **Subtitle-only downloads** (`skip_download=True`): Deno NOT required.
- **Video downloads** (`skip_download=False`): Deno IS required.

### Windows (PowerShell)
```powershell
irm https://deno.land/install.ps1 | iex
```
Then restart Jupyter so the new PATH is picked up.

### macOS / Linux
```bash
curl -fsSL https://deno.land/install.sh | sh
```

In [ ]:
# Check if Deno is installed (informational only)
import shutil
deno_path = shutil.which('deno')
if deno_path:
    print(f'✅ Deno found at: {deno_path}')
    import subprocess
    print(subprocess.run(['deno', '--version'], capture_output=True, text=True).stdout)
else:
    print('⚠️ Deno not found. Install it (Step 3) if you plan to download VIDEO files.')
    print('   For subtitle-only downloads (skip_download=True), Deno is not required.')

## Step 4: Cookies

YouTube requires authentication to avoid bot detection.
Without cookies, you will get: **"Sign in to confirm you're not a bot"**

### How to get cookies.txt:
1. Install a browser extension to export cookies:
   - **Chrome**: [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc)
   - **Firefox**: [cookies.txt](https://addons.mozilla.org/en-US/firefox/addon/cookies-txt/)
2. Log into YouTube in your browser
3. Click the extension and export cookies for youtube.com → save as `cookies.txt`
4. Place `cookies.txt` next to this notebook, or set an absolute path below

In [ ]:
import os
# Path to cookies.txt. Place it next to this notebook, or use an absolute path.
cookies_path = 'cookies.txt'
if os.path.exists(cookies_path):
    print(f'✅ Cookies found at: {os.path.abspath(cookies_path)}')
else:
    print(f'⚠️ cookies.txt NOT found at: {os.path.abspath(cookies_path)}')
    print('   YouTube may return "Sign in to confirm you\'re not a bot" without cookies.')

## Step 5: Import & Save Path

In [ ]:
from gorendir import YouTubeDownloader
print('✅ gorendir imported successfully')

In [ ]:
# Local save directory. Use an absolute path for clarity.
# Windows example: r'C:\Users\YourName\Videos\YouTubeDownloads'
# macOS / Linux example: '/home/yourname/Videos/YouTubeDownloads'
save_path = r'./youtube_downloads'
import os; os.makedirs(save_path, exist_ok=True)
print(f'Save path: {os.path.abspath(save_path)}')

## Step 6: Helper Function

The `download_videos` function wraps `YouTubeDownloader.download_video()`.

### Input Format for `video_urls`

| Format | Example | Description |
|--------|---------|-------------|
| `str` | `"https://youtube.com/watch?v=XXX"` | Single URL |
| `Dict[str, int]` | `{"https://youtube.com/playlist?list=XXX": 8}` | URL with start number |
| `List[Union[str, Dict[str, int]]]` | `["url1", {"url2": 5}]` | Mixed list |

When using `Dict[str, int]`, the integer value means:
- **Skip** the first N-1 videos in download order
- **Number** files starting from that number

> **Note:** We do NOT pass `verify_ssl` here because SSL is already disabled globally in Step 2 via monkey-patching. This keeps the notebook compatible with any gorendir version.

In [ ]:
def download_videos(video_urls, save_path, reverse_download=False, skip_download=False,
                    playlist_end=0, cookies_path=None):
    """
    Download videos using GörEndir.

    Args:
        video_urls: URL(s) to download. Can be:
            - str: Single URL
            - Dict[str, int]: {"url": start_number}  e.g. {"https://...playlist?list=XXX": 8}
            - List[Union[str, Dict[str, int]]]: Mixed list of URLs and URL-start_number dicts
        save_path: Directory to save downloads
        reverse_download: If True, reverse playlist order (last video -> #1)
        skip_download: If True, only download subtitles (skip video file)
        playlist_end: If > 0, stop after downloading this many videos
        cookies_path: Path to cookies.txt for YouTube authentication

    Note:
        SSL verification is disabled globally in Step 2 via monkey-patching,
        so no `verify_ssl` argument is needed here. This keeps the function
        compatible with any installed version of gorendir.
    """
    downloader = YouTubeDownloader(
        save_directory=save_path,
        max_resolution=1080,
        subtitle_languages=["en", "fa"],
        cookies_path=cookies_path,
    )

    downloader.download_video(
        video_urls=video_urls,
        reverse_download=reverse_download,
        skip_download=skip_download,
        force_download=False,
        yt_dlp_write_subs=True,
        download_subtitles=True,
        playlist_end=playlist_end,
    )

## Step 7: Download Configurations

Define different download scenarios. Each config has:
- `name`: A label for display
- `reverse_download`: Whether to reverse playlist order
- `skip_download`: Whether to skip video (only download subtitles)
- `playlist_end`: Limit number of videos (0 = no limit)
- `urls`: The URL(s) - can be a **list of strings**, a **dict of url→start_number**, or a **mixed list**

In [ ]:
# -- Subtitle-only (reverse order) --
sbtitle_reverse_urls = {
    'name': 'sbtitle_reverse_urls',
    'reverse_download': True,
    'skip_download': True,
    'playlist_end': 0,
    'urls': [
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 8},
    ]
}

In [ ]:
# -- Video + Subtitle (reverse order) --
video_reverse_urls = {
    'name': 'video_reverse_urls',
    'reverse_download': True,
    'skip_download': False,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
    ]
}

In [ ]:
# -- Subtitle-only (normal order) --
sbtitle_urls = {
    'name': 'sbtitle_urls',
    'reverse_download': False,
    'skip_download': True,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 3},
    ]
}

In [ ]:
# -- Video + Subtitle (normal order) --
video_urls_config = {
    'name': 'video_urls',
    'reverse_download': False,
    'skip_download': False,
    'playlist_end': 0,
    'urls': [
        # "https://www.youtube.com/watch?v=_S_af9C9z2Q",
    ]
}

## Step 8: Run Downloads

In [ ]:
video_list = [sbtitle_reverse_urls, video_reverse_urls, sbtitle_urls, video_urls_config]

for config in video_list:
    print('*' * 20, config['name'], '*' * 20)
    urls = config['urls']
    if not urls:
        print(f"  No URLs configured for {config['name']}, skipping.")
        continue

    download_videos(
        video_urls=urls,
        save_path=save_path,
        reverse_download=config['reverse_download'],
        skip_download=config['skip_download'],
        playlist_end=config.get('playlist_end', 0),
        cookies_path=cookies_path,
    )

---

## Advanced Usage Examples

In [ ]:
# Example 1: Single video
# download_videos("https://youtube.com/watch?v=dQw4w9WgXcQ", save_path,
#                 cookies_path=cookies_path)

In [ ]:
# Example 2: Dict format — skip first 7, start numbering from 8
# download_videos({"https://youtube.com/playlist?list=PLxxxxxxx": 8}, save_path,
#                 cookies_path=cookies_path)

In [ ]:
# Example 3: Mixed list
# download_videos([
#     "https://youtube.com/watch?v=abc123",
#     {"https://youtube.com/playlist?list=PLxxxx": 5},
# ], save_path, cookies_path=cookies_path)

In [ ]:
# Example 4: Reverse + start number
# download_videos(
#     {"https://youtube.com/playlist?list=PLxxxx": 8},
#     save_path,
#     reverse_download=True,
#     cookies_path=cookies_path
# )

In [ ]:
# Example 5: Limit to 10 videos
# download_videos(
#     "https://youtube.com/playlist?list=PLxxxx",
#     save_path,
#     playlist_end=10,
#     cookies_path=cookies_path
# )

In [ ]:
# Example 6: YouTubeDownloader directly with all options
# downloader = YouTubeDownloader(
#     save_directory=save_path,
#     max_resolution=720,
#     subtitle_languages=["en", "fa", "tr"],
#     retry_attempts=5,
#     cookies_path=cookies_path,
# )
# downloader.download_video(
#     video_urls={"https://youtube.com/playlist?list=PLxxxx": 3},
#     reverse_download=False,
#     skip_download=False,
#     force_download=True,
#     yt_dlp_write_subs=True,
#     download_subtitles=True,
#     playlist_end=0,
# )